# Airbnb Paris – Experiment 2: Enhanced Baseline-Modelle
- Gleiche Detektoren auf verschiedenen Repräsentationen: semantisch, fastText, enhanced, enhanced+semantisch
- Grid Search **je Repräsentation** auf Val (AUPRC); `cleaned` entfällt hier, das deckt Experiment 1 ab
- Alignment über `row_id`, Split aus der Split-Datei

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Repräsentationen laden
- `enhanced_semantic_pca30` wird hier aus `enhanced_pca30` + `semantic_pca30` zusammengesetzt

In [ ]:
names = ["semantic_pca100", "semantic_pca30", "fast_text_pca100", "fast_text_pca30", "enhanced", "enhanced_pca30"]
reps = {n: pd.read_csv(f"../../data/preprocessed/{n}_airbnb_paris_seed{SEED}.csv").set_index("row_id").drop(columns=["is_top_rating"])
        for n in names}
reps["enhanced_semantic_pca30"] = reps["enhanced_pca30"].join(reps["semantic_pca30"], how="inner",
                                                              lsuffix="_enh", rsuffix="_sem")
print({n: r.shape[1] for n, r in reps.items()})

## Gemeinsamer Index, Label & Split

In [ ]:
cleaned = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv").set_index("row_id")
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv").set_index("row_id")

common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()

y = (1 - cleaned.loc[common, "is_top_rating"]).values
s = split.loc[common, "split"].values
y_tr, y_va, y_te = y[s == "train"], y[s == "val"], y[s == "test"]
k = round(len(y_te) * y_tr.mean())
print("Zeilen:", len(common), "| train/val/test:", (s == "train").sum(), (s == "val").sum(), (s == "test").sum())

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_2")

## Detektoren × Repräsentationen
- Je Kombination eigene Grid Search auf Val, Refit auf Train, Auswertung auf Test

In [ ]:
detectors = {
    "iforest": (IForest, {"n_estimators": [100, 200], "max_features": [0.5, 1.0], "random_state": [SEED]}),
    "loda": (LODA, {"n_bins": [10, 20], "n_random_cuts": [100, 200]}),
    "ecod": (ECOD, {}),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [[64, 32], [32, 16]], "epoch_num": [20, 50],
                                  "random_state": [SEED]}),
}

for rep_name, rep in reps.items():
    M = rep.loc[common].values
    X_tr, X_va, X_te = M[s == "train"], M[s == "val"], M[s == "test"]
    for det_name, (Model, grid) in detectors.items():
        t0 = time.perf_counter()
        best_auprc, best_params = -1.0, {}
        for params in (list(ParameterGrid(grid)) or [{}]):
            m = Model(**params)
            m.fit(X_tr)
            p, r, _ = precision_recall_curve(y_va, m.decision_function(X_va))
            val_auprc = auc(r, p)
            if val_auprc > best_auprc:
                best_auprc, best_params = val_auprc, params

        model = Model(**best_params)
        model.fit(X_tr)
        scores = model.decision_function(X_te)
        runtime = time.perf_counter() - t0

        prec, rec, _ = precision_recall_curve(y_te, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_te, scores)
        pred = (scores >= np.sort(scores)[-k]).astype(int)

        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_params({**best_params, "representation": rep_name, "detector": det_name,
                               "n_features": rep.shape[1], "seed": SEED})
            mlflow.log_metric("auprc", auprc)
            mlflow.log_metric("auc_roc", auroc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")
        print(classification_report(y_te, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))